- Find the name of the user who has rated the greatest number of movies. In case of a tie, return the lexicographically smaller user name.
- Find the movie name with the highest average rating in February 2020. In case of a tie, return the lexicographically smaller movie name.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

In [0]:
#creating movies dataframe
movies_data = [(1,'Avengers'),(2,'Frozen 2'),(3,'Joker')]
movies_cols =["movie_id","title"]

movies_df = spark.createDataFrame(movies_data,movies_cols)
display(movies_df)

In [0]:
# Creating users dataframe
users_data = [(1,'Daniel'),(2,'Monica'),(3,'Maria'),(4,'James')]
users_cols = ("user_id","name")
users_df = spark.createDataFrame(users_data,users_cols)
display(users_df)

In [0]:
#creating movie rating data
movie_rating_data =[(1,1,3,'2020-01-12'),(1,2,4,'2020-01-11'),(1,3,2,'2020-02-12')]
movie_rating_cols =["movie_id","user_id","rating","created_at"]
movie_rating_df = spark.createDataFrame(movie_rating_data,movie_rating_cols)
display(movie_rating_df)

In [0]:
#️⃣ retrieving the name of the user who rated the most movies
joined_df = users_df.join(movie_rating_df, on = "user_id", how = "left")\
.groupBy("user_id", "name")\
.agg(F.count("movie_id").alias("ratings"))\
.orderBy(["ratings", "name"], ascending=[False, True]).limit(1)
display(joined_df)

In [0]:
#️⃣ retrieving the movie with the highest average rating in february 2020
result_df = movie_rating_df.join(movies_df, on = "movie_id", how = "inner")\
.filter(F.col("created_at").between("2020-02-01", "2020-02-29"))\
.groupBy("movie_id", "title").agg(F.avg("rating").alias("avg_rating"))\
.orderBy(["avg_rating","title"], ascending = [False, True])\
.select("title").limit(1)
display(result_df)

In [0]:
#️⃣ retrieving top user and top movie
top_user_name = joined_df.take(1)[0]["name"]
top_movie_title = result_df.take(1)[0]["title"]

#️⃣ creating a new dataframe and combining both results
final_df = spark.createDataFrame([(top_user_name,),(top_movie_title,)], ["result"])

#️⃣ displaying the final result
final_df.show()
